In [0]:
%sql
CREATE OR REPLACE TABLE de_mini_project.03_gold_schema.kpi_aov AS
SELECT 
  ROUND(COALESCE(SUM(order_revenue) / NULLIF(COUNT(DISTINCT order_id), 0), 0), 2) AS aov
FROM (
  SELECT order_id, SUM(revenue) AS order_revenue
  FROM de_mini_project.03_gold_schema.curated_orders
  GROUP BY order_id
);


In [0]:
%sql
-- 2-kpi Delivery Time 
SELECT ROUND(AVG(delivery_days),2) AS avg_days
FROM  de_mini_project.03_gold_schema.curated_orders;

In [0]:
%sql
-- 3-kpi KPI_CLV 
SELECT customer_id, ROUND(SUM(revenue),2) clv
FROM de_mini_project.03_gold_schema.curated_orders
GROUP BY customer_id;

In [0]:
%sql
--4-kpi late-delivery 
SELECT ROUND(AVG(is_late_delivery)*100,2) AS late_delivery_percent
FROM de_mini_project.03_gold_schema.curated_orders;

In [0]:
%sql
-- 5-kpi low ratings
SELECT
SUM(CASE WHEN review_score < 3 THEN 1 ELSE 0 END)*100.0/COUNT(*) AS low_rating_pct
FROM de_mini_project.03_gold_schema.curated_orders;

In [0]:
%sql
-- 6 kpi customer new returning
WITH first_order AS (
SELECT customer_id, MIN(order_date) first_date
FROM  de_mini_project.03_gold_schema.curated_orders GROUP BY customer_id
)
SELECT
year, month,
COUNT(DISTINCT CASE WHEN order_date = first_date THEN c.customer_id END) new_customers,
COUNT(DISTINCT CASE WHEN order_date > first_date THEN c.customer_id END) returning_customers
FROM  de_mini_project.03_gold_schema.curated_orders c
JOIN first_order f ON c.customer_id=f.customer_id
GROUP BY year, month;

In [0]:
%sql
-- 7-kpi fulfillment
SELECT
COUNT(CASE WHEN order_status='delivered' THEN 1 END)*100.0/COUNT(*) AS fulfillment_rate
FROM de_mini_project.03_gold_schema.curated_orders;

In [0]:
%sql
-- 8 -kpi orders by location
SELECT city, state, COUNT(DISTINCT order_id) total_orders
FROM de_mini_project.03_gold_schema.curated_orders
GROUP BY city, state;

In [0]:
%sql
-- 9 -kpi payment distribution
SELECT 
payment_type,
COUNT(*) AS cnt,
ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),2) AS percentage
FROM de_mini_project.03_gold_schema.curated_orders
WHERE payment_type IS NOT NULL
GROUP BY payment_type;

In [0]:
%sql
-- 10 -kpi revenue monthly
SELECT year, month, ROUND(SUM(revenue),2) AS total_revenue
FROM de_mini_project.03_gold_schema.curated_orders
GROUP BY year, month;

In [0]:
%sql
-- 11 -kpi revenue by product
SELECT product_category_name, ROUND(AVG(review_score),2) avg_rating
FROM de_mini_project.03_gold_schema.curated_orders
GROUP BY product_category_name;

In [0]:
%sql
-- 12 -kpi revenue by seller
SELECT seller_id, ROUND(AVG(review_score),2) as avg_rating
FROM de_mini_project.03_gold_schema.curated_orders
GROUP BY seller_id;

In [0]:
%sql
-- 13 -kpi top products
SELECT product_category_name, SUM(revenue) AS revenue
FROM de_mini_project.03_gold_schema.curated_orders
GROUP BY product_category_name
ORDER BY revenue DESC
LIMIT 10;

In [0]:
%sql
-- 14 -kpi top seller
SELECT seller_id, ROUND(SUM(revenue),2) revenue
FROM de_mini_project.03_gold_schema.curated_orders
GROUP BY seller_id
ORDER BY revenue DESC LIMIT 10;